# TN1 — Kiến trúc: TCN so với DS-TCN

Đổi **đúng một biến** so với TN0: thay LSTM của MobiVital bằng TCN. Mọi thứ khác giữ nguyên cấu hình tác giả công bố — 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN (để dành TN2).

## Câu hỏi

**Thu nhỏ model 90–96% thì còn dự báo tốt hơn LSTM không?**

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| LSTM (MobiVital) | hidden 352 | 1.502.713 | — |
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |

Ba cấu hình, mỗi cấu hình 4 fold, một seed. Khoảng 3.7 giờ trên L4.

Câu hỏi *"cho ngang lượng tham số thì kiến trúc nào hơn"* để dành cho thí nghiệm riêng — TN1 chỉ đổi đúng một biến là kiến trúc, ở cùng một mức kênh.

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

**`G H I J` không được đụng tới.** Chúng chỉ dùng ở bước công bố cuối, sau khi mọi cấu hình đã chốt — xem `docs/PROTOCOL.md`.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật. Cửa sổ train thì có lọc bằng `corr > 0.9`, nên không được dùng cửa sổ để chấm.

## Cơ sở chọn tham số kiến trúc

`kernel=3`, `n_blocks=6`, hai tầng conv mỗi khối, `dropout=0.0` — mỗi số đều trích dẫn được, xem [`docs/THAM_CHIEU.md`](../docs/THAM_CHIEU.md). Ràng buộc quan trọng nhất: **tầm nhìn phải phủ hết 200 mẫu vào**; `k=3, n=6` cho 253 mẫu.

Kết quả vào `runs/tn1/`, cuối notebook nén thành `runs/tn1.zip`.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo họ.


In [ ]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


Lấy `by_user/` và `windows/` từ Drive. TN1 **không cần CSV thô 13 GB** — chỉ train trên cửa sổ đã cắt và chấm trên `by_user/*.npz`.


In [ ]:
!python scripts/restore_processed_data_on_drive.py


## 2. Đo tốc độ trước khi chạy

Chạy hết TN1 mất vài giờ. Đo trước một batch của từng kiến trúc để ước đúng thời gian, và để trả lời câu "TCN nhẹ hơn thì có nhanh hơn không".

Đo trên dữ liệu ngẫu nhiên đúng kích thước thật — tốc độ chỉ phụ thuộc hình dạng tensor.


In [ ]:
!python scripts/do_toc_do.py


## 3. Ba cấu hình

Mỗi lệnh chạy trọn 4 fold rồi ghi 5 dòng vào `runs/summary.csv`: bốn dòng fold và một dòng `TONG` mang `cv_score` cùng `cv_std`.


**LSTM** — mốc so sánh. Phải chạy trong CV, vì số LSTM ở TN0 đo trên `G H I J` chứ không trên fold.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model lstm


**TCN-64** — tích chập nhân quả giãn dần, 151.513 tham số, nhẹ hơn LSTM 90%.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64


**DS-TCN-64** — tách depthwise và pointwise, 56.281 tham số, nhẹ hơn LSTM 96%.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64


Kết quả sơ bộ sau ba cấu hình:


In [ ]:
!python scripts/compare_cv.py --experiment tn1 --so-doi lstm


## 5. So sánh

Hai bảng:

1. **cv_score** — điểm trung bình 4 fold của từng cấu hình, kèm chi tiết từng fold và độ lệch chuẩn
2. **thắng / hoà / thua** — so **từng** buổi ghi với LSTM, trên đủ 1289 buổi của tám người dev

Bảng 2 cần thiết vì hai cấu hình chênh nhau 0.003 điểm trung bình có thể là tốt hơn đều khắp, hoặc thắng đậm vài buổi mà thua nhẹ phần lớn. Trung bình không phân biệt được.


In [ ]:
!python scripts/compare_cv.py --experiment tn1 --so-doi lstm


## 5b. Mốc kiểm chứng trên G H I J

Đến đây `cv_score` đã chọn xong kiến trúc. Chạy thêm một mốc để biết **có đi đúng hướng không**: train đủ tám người `A B C D E F K L` rồi test 537 buổi ghi của `G H I J` — đúng pipeline mà bài báo dùng.

Ba seed mỗi cấu hình, báo cáo **mean ± std**. Một lần chạy cho một con số không nói được gì: trọng số khởi tạo ngẫu nhiên và thứ tự xáo trộn dữ liệu đổi theo seed, hai cấu hình chênh nhau 0.005 có thể chỉ là may rủi.

Chạy cho **hai** cấu hình: kiến trúc thắng ở mục 5, và **LSTM** làm mốc. LSTM cũng phải train đủ tám người, cùng pipeline, mới công bằng.

**Đây chưa phải số công bố.** Nếu TN2–TN6 đổi cấu hình thì con số này thành cũ và phải chạy lại ở bước cuối. Nó chỉ là mốc kiểm chứng của TN1.


In [ ]:
# LSTM làm mốc, 3 seed
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 2


Kiến trúc thắng ở mục 5 — chạy **một** trong hai ô dưới:


In [ ]:
# chạy ô này nếu TCN thắng
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 2


In [ ]:
# chạy ô này nếu DS-TCN thắng
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 2


Gộp ba seed thành `mean ± std`:


In [ ]:
!python scripts/compare_cv.py --experiment tn1_ghij --final


## 6. Lưu kết quả

Nén `runs/tn1/` và `runs/tn1_ghij/` thành hai tệp zip, chép sang Drive. Giải nén lại bằng `unzip tn1.zip -d runs/`.


In [ ]:
!python scripts/save_results.py tn1
!python scripts/save_results.py tn1_ghij
